In [64]:
from concurrent.futures import ThreadPoolExecutor, as_completed
import re
import requests
import datetime
from bs4 import BeautifulSoup
from python_utils import converters
import time
import zoneinfo
import tzlocal
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.firefox.options import Options
import time
from datetime import datetime
from dateutil import parser
import pprint
import pandas as pd
import datetime
import os
import json
import re
import random
from collections import OrderedDict
import pyperclip
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.common.keys import Keys
import urllib.request



In [65]:
HLTV_COOKIE_TIMEZONE = "Europe/Copenhagen"
HLTV_ZONEINFO=zoneinfo.ZoneInfo(HLTV_COOKIE_TIMEZONE)
LOCAL_TIMEZONE_NAME = tzlocal.get_localzone_name()
LOCAL_ZONEINFO = zoneinfo.ZoneInfo(LOCAL_TIMEZONE_NAME)

TEAM_MAP_FOR_RESULTS = []
def _get_all_teams():
    if not TEAM_MAP_FOR_RESULTS:
        teams = get_parsed_page("https://www.hltv.org/stats/teams?minMapCount=0")
        for team in teams.find_all("td", {"class": ["teamCol-teams-overview"], }):
            team = {'id': converters.to_int(team.find("a")["href"].split("/")[-2]), 'name': team.find("a").text, 'url': "https://hltv.org" + team.find("a")["href"]}
            TEAM_MAP_FOR_RESULTS.append(team)

def _findTeamId(teamName: str):
    _get_all_teams()
    for team in TEAM_MAP_FOR_RESULTS:
        if team['name'] == teamName:
            return team['id']
    return None

def _padIfNeeded(numberStr: str):
    if int(numberStr) < 10:
        return str(numberStr).zfill(2)
    else:
        return str(numberStr)

def _monthNameToNumber(monthName: str):
    # Check for the input "Augu" and convert it to "August"
    # This is necessary because the input string may have been sanitized
    # by removing the "st" from the day numbers, such as "21st" -> "21"
    if monthName == "Augu":
        monthName = "August"
    return datetime.datetime.strptime(monthName, '%B').month

def get_parsed_page(url):
    # Set up the Firefox options for headless mode
    options = Options()
    options.add_argument("--headless")
    
    # Initialize the Firefox WebDriver
    driver = webdriver.Firefox(options=options)
    
    # Navigate to the URL
    driver.get(url)

    # Delay to allow the page to load fully
    time.sleep(random.uniform(3, 5))  # Random delay between 1-3 seconds

    try:
        # Find and click the 'Accept Cookies' button
        cookie_button = driver.find_element(By.XPATH, '//*[@id="CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll"]')
        cookie_button.click()

        # Delay after clicking the cookie consent button to allow the page to reload
        #time.sleep(random.uniform(1, 2))  # Random delay between 1-2 seconds
    except Exception as e:
        print(f"Error clicking cookie consent: {e}")
    driver.save_screenshot("datacamp.png")

    # Get the page source after interaction
    html_source = driver.page_source

    # Parse the page source with BeautifulSoup
    soup = BeautifulSoup(html_source, 'html.parser')
    
    # Quit the WebDriver session after scraping
    driver.quit()

    return soup

 #Function to run multiple URLs concurrently

def scrape_urls_concurrently(urls, max_workers=4):
    results = []
    i = 1
    # Use ThreadPoolExecutor to handle multiple threads
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        # Submit all URLs as tasks to the executor
        futures = {executor.submit(get_parsed_page, url): url for url in urls}
        
        for future in as_completed(futures):
            url = futures[future]
            try:
                soup = future.result()  # Get the BeautifulSoup object from the task
                print_str = f"({i}/{len(urls)}) - Successfully scraped {url}"
                print(print_str)
                results.append([soup, url])
                i+=1
                # Global delay between requests to avoid rate limiting
                time.sleep(random.uniform(2, 5))  # Random delay between 2-5 seconds

            except Exception as e:
                print(f"Error scraping {url}: {e}")
        executor.shutdown(wait=True)
    return results


#Players

def get_player_url(player_id, nickname, match_type=None, startDate=None, endDate=None, ranking=None, map_name=None):
    # Base URL
    url = f"https://www.hltv.org/stats/players/{player_id}/{nickname}"
    
    # Initialize a list to hold query parameters
    query_params = []
    
    # Add optional parameters if they are provided
    if match_type:
        query_params.append(f"matchType={match_type}")
    
    if startDate and endDate:
        query_params.append(f"startDate={startDate}&endDate={endDate}")
    
    if ranking:
        query_params.append(f"rankingFilter={ranking}")
    
    if map_name:
        query_params.append(f"maps={map_name}")
    
    # If there are any query parameters, append them to the URL
    if query_params:
        url += "?" + "&".join(query_params)

    return url

def get_player_stats(player_id, nickname, match_type=None, startDate=None, endDate=None, ranking=None, map_name=None):
    return get_player_stats_sup(get_parsed_page(get_player_url(player_id, nickname, match_type, startDate, endDate, ranking, map_name)))

def get_player_stats_sup(soup):
    
    # Initialize an OrderedDict to hold player stats in the desired order
    player_stats = OrderedDict()

    # Player Summary Stats (from "playerSummaryStatBox")
    summary_stats = OrderedDict()  # Also using OrderedDict for consistent order in summary stats
    summary_stat_box = soup.find('div', class_='playerSummaryStatBox')
    if not summary_stat_box:
        print ("No player summary stats found")
        return
    info = summary_stat_box.find('div', class_='summaryShortInfo')
    

    # Populate player details first
    player_stats['name'] = info.find('div', class_='summaryRealname text-ellipsis').text.strip()
    player_stats['nickname'] = info.find('h1', class_='summaryNickname text-ellipsis').text.strip()

    team_element = info.find('div', class_='SummaryTeamname text-ellipsis').find('a', class_='a-reset text-ellipsis')
    if team_element is not None:
        player_stats['team'] = team_element.text.strip()
    else:
        player_stats['team'] = 'No Team'
    player_stats['age'] = info.find('div', class_='summaryPlayerAge').text.strip().split(" ")[0]
    player_stats['country'] = info.find('img', class_='flag').get('title')
    
    teammates = []
    teammate_elements = soup.find_all('div', class_='teammate standard-box')

    for teammate_element in teammate_elements:
        teammate_info = OrderedDict()

        name_and_nickname = teammate_element.find('img').get('title').split("'")
        teammate_info['full_name'] = name_and_nickname[0].strip()
        teammate_info['nickname'] = name_and_nickname[1].strip()

        rating_element = teammate_element.find('div', class_='teammate-info')  # Update based on HTML structure
        text = rating_element.text.strip().split("\n")
        teammate_info['rating'] = text[len(text) - 1]
        
        teammates.append(teammate_info)

    player_stats['teammates'] = teammates

    # Player summary stats - NOT WORKING BUT TOTAL STATS INCLUDES
    # summary_row = summary_stat_box.find_all('div', class_='summaryStatBreakdownRow')
    # for row in summary_row:
    #     stats = row.find_all('div', class_='summaryStatBreakdown aboveAverage')
    #     for stat in stats:
    #         stat_name = stat.find('div', class_='summaryStatBreakdownSubHeader').contents[0].strip()
    #         stat_value = stat.find('div', class_='summaryStatBreakdownDataValue').text.strip()
    #         summary_stats[stat_name] = stat_value
            
    # player_stats['summary_stats'] = summary_stats

    
    # Player Total Stats (from "statistics")
    total_stats = soup.find('div', class_='statistics')
    total_result = OrderedDict()

    for stat in total_stats.find_all('div', class_='stats-row'):
        inside = stat.find_all('span')
        stat_name = inside[0].text.strip()
        stat_value = inside[1].text.strip()
        total_result[stat_name] = stat_value

    player_stats["total_stats"] = total_result

    # Higher-level headers: "Combined", "CT", "T"
    sides = ['Combined', 'CT', 'T']
    role_stats = OrderedDict()

    # Initialize each side with an OrderedDict to hold subheaders and their stats
    for side in sides:
        role_stats[side] = OrderedDict()

    # Find all role stats sections (Firepower, Clutching, etc.)
    role_stats_section = soup.find_all('div', class_='role-stats-section')

    for section in role_stats_section:
        # Each section has a subheader (e.g., Firepower, Damage, etc.)
        subheader = section.find('div', class_='role-stats-section-title').text.strip().split('\n')[0]

        # Stat rows for this section (each section has stats for Combined, CT, and T)
        stat_rows = section.find_all('div', class_='role-stats-row')

        # Loop through the stat rows and assign to Combined, CT, and T
        for index, stat_row in enumerate(stat_rows):
            # Determine which side (Combined, CT, T) based on the index
            side = sides[index % len(sides)]

            # If subheader is not yet present under this side, initialize it
            if subheader not in role_stats[side]:
                role_stats[side][subheader] = OrderedDict()

            # Extract the stat name and value
            stat_name = stat_row.find('div', class_='role-stats-title').text.strip()
            stat_value = stat_row.find('div', class_='role-stats-data').text.strip()

            # Store the stat under the correct side and subheader
            role_stats[side][subheader][stat_name] = stat_value

    # Assign the role stats to player_stats under 'role_stats'
    player_stats['role_stats'] = role_stats

    return player_stats
                    
def get_top_player_stats():
    players = top_players()
    urls = []

    for player in players:
        urls.append(get_player_url(player['id'], player['nickname']))
    
    soups = scrape_urls_concurrently(urls)

    try:
        results = []
        for soup in soups:
            results.append(get_player_stats_sup(soup[0]))
            time.sleep(random.uniform(3, 7))  # Random delay between 2-5 seconds
    except:
        print("Failed on link: " + soup[1])

    return results

def top_players():
    page = get_parsed_page("https://www.hltv.org/stats")
    players = page.find_all("div", {"class": "col"})[0]
    playersArray = []
    for player in players.find_all("div", {"class": "top-x-box standard-box"}):
        playerObj = {}
        playerObj['country'] = player.find_all('img')[1]['alt']
        buildName = player.find('img', {'class': 'img'})['alt'].split("'")
        playerObj['name'] = buildName[0].rstrip() + buildName[2]
        playerObj['nickname'] = player.find('a', {'class': 'name'}).text
        playerObj['rating'] = player.find('div', {'class': 'rating'}).find('span', {'class': 'bold'}).text
        playerObj['maps-played'] = player.find('div', {'class': 'average gtSmartphone-only'}).find('span', {'class': 'bold'}).text
        playerObj['url'] = "https://hltv.org" + player.find('a', {'class': 'name'}).get('href')
        playerObj['id'] = converters.to_int(player.find('a', {'class': 'name'}).get('href').split("/")[-2])
        playersArray.append(playerObj)
    return playersArray

def get_players_link(map=None, matchtype=None, level=None, startDate="all", endDate=None, side=None):
    url = "https://www.hltv.org/stats/players?&startDate=" + startDate
    if map:
        url += "&maps=" + map
    if matchtype:
        url += "&matchType=" + matchtype
    if endDate:
        url += "&endDate=" + endDate
    if side:
        url += "&side=" + side
    if level:
        url += "&rankingFilter=Top" + str(level)
    return url

def get_players(map=None, matchtype=None, count=None, startDate="all", endDate=None, side=None, level=None):
    url = get_players_link(map, matchtype, level, startDate, endDate, side)
    page = get_parsed_page(url)

    playersList = {}
    i = 0

    players = page.find("table", {"class": "stats-table player-ratings-table"})
    playersArray = players.find("tbody").find_all("tr")
    for player in playersArray:
        
        if i == count:
            return playersList

        playerObj = OrderedDict()
        playerObj['country'] = player.find_all('img')[1]['alt']
        nickname = player.text.strip().split("\n")[0]
        teamCol =  player.find('td', {'class': 'teamCol'})
        playerObj['team'] = []
        for img in teamCol.find_all('img'): playerObj['team'].append(img.get('title'))
        playerObj['maps'] = player.find('td', {'class': 'statsDetail'}).text.strip()
        playerObj['rounds'] = player.find('td', {'class': 'statsDetail gtSmartphone-only'}).text.strip()
        playerObj['K-D Diff'] = player.find('td', {'class': lambda value: value and "kdDiffCol" in value.split()}).text.strip()
        playerObj['Rating'] = player.find('td', {'class': lambda value: value and "ratingCol" in value.split()}).text.strip()
        playerObj['Id'] = player.find('a').get('href').split("/")[3]
        playersList[nickname] = playerObj
        i += 1


    return playersList

def get_players_stats(map=None, matchtype=None, count=None, startDate="all", endDate=None, side=None, level=None, append=False):
    players = get_players(map, matchtype, count, startDate, endDate, side, level)
    urls = []
    player_stats_list = []

    for player in players:
        urls.append(get_player_url(players[player]['Id'], player))

    i = 1

    if append:
        # Load existing data if the file exists
        if os.path.exists("Outputs/results.json"):
            with open("Outputs/results.json", 'r') as json_file:
                try:
                    existing_data = json.load(json_file)
                except json.JSONDecodeError:
                    existing_data = []  # If the file is empty or not a valid JSON
        else:
            existing_data = []

        for url in urls:
            try:
                current_soup = get_parsed_page(url)
                print(f"({i}/{len(urls)}) - Successfully scraped {url}")
                try:
                    player_stats = get_player_stats_sup(current_soup)
                    print(f"Stats for {i} have been found")
                    
                    # Append the current player's stats to the existing data
                    existing_data.append(player_stats)
                    
                except Exception as e:
                    print(f"Failed on link {url}: {e}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")
                
            time.sleep(random.uniform(4, 10))
            i += 1

        # Write the updated data back to the file
        with open("Outputs/results.json", 'w') as json_file:
            json.dump(existing_data, json_file, indent=4)
    else:
        for url in urls:
            try:
                current_soup = get_parsed_page(url)
                print(f"({i}/{len(urls)}) - Successfully scraped {url}")
                try:
                    player_stats = get_player_stats_sup(current_soup)
                    print(f"Stats for {i} have been found")
                    player_stats_list.append(player_stats)
                except Exception as e:
                    print(f"Failed on link {url}: {e}")
            except Exception as e:
                print(f"Error scraping {url}: {e}")

            time.sleep(random.uniform(4, 10))
            i += 1

    if not append:
        return player_stats_list

#Matches

def get_results(stars=None, limit=None, startDate=None, endDate=None):    
    urls = get_matches_url(limit, startDate, endDate, stars)
    
    soups = scrape_urls_concurrently(urls[:limit])
    results_list = []

    back = 100
    
    if(limit and limit < 100):
        back = limit

    i = 0

    for soup in soups:
        
        #grabbing soup because it stores link and soup
        soup = soup[0]
        pastresults = soup.find_all("div", {"class": "results-holder"})
    
        for result in pastresults:
            resultDiv = result.find_all("div", {"class": "result-con"})

            header = result.find("div", {"class": "results-all"})
            if header:
                dateText = re.sub(r'(\d+)(th|st|nd|rd)', r'\1', header.text.replace("Results for ", ""))

                dateArr = dateText.split()
                dateTextFromArrPadded = _padIfNeeded(dateArr[2]) + "-" + _padIfNeeded(_monthNameToNumber(dateArr[0])) + "-" + _padIfNeeded(dateArr[1])

                dateFromHLTV = datetime.datetime.strptime(dateTextFromArrPadded, '%Y-%m-%d').replace(tzinfo=HLTV_ZONEINFO)
                dateFromHLTV = dateFromHLTV.astimezone(LOCAL_ZONEINFO)

                formatted_date = dateFromHLTV.strftime('%m/%d/%Y')
                formatted_date = formatted_date.lstrip('0').replace('/0', '/')

            else:
                dt = datetime.date.today()
                formatted_date = str(dt.month) + '/' + str(dt.day) + '/' + str(dt.year)

            resultDiv = result.find_all("div", {"class": "result-con"})  
            
            for res in resultDiv:

                if back < 100 and i == back:
                    return results_list

                resultObj = {}

                resultObj['url'] = "https://hltv.org" + res.find("a", {"class": "a-reset"}).get("href")
                
                resultObj['match-id'] = converters.to_int(res.find("a", {"class": "a-reset"}).get("href").split("/")[-2])
                
                resultObj['date'] = formatted_date

                if (res.find("td", {"class": "placeholder-text-cell"})):
                    resultObj['event'] = res.find("td", {"class": "placeholder-text-cell"}).text
                elif (res.find("td", {"class": "event"})):
                    resultObj['event'] = res.find("td", {"class": "event"}).text
                else:
                    resultObj['event'] = None

                if (res.find_all("td", {"class": "team-cell"})):
                    resultObj['team1'] = res.find_all("td", {"class": "team-cell"})[0].text.lstrip().rstrip()
                    resultObj['team1score'] = converters.to_int(res.find("td", {"class": "result-score"}).find_all("span")[0].text.lstrip().rstrip())
                    resultObj['team1-id'] = _findTeamId(res.find_all("td", {"class": "team-cell"})[0].text.lstrip().rstrip())
                    resultObj['team2'] = res.find_all("td", {"class": "team-cell"})[1].text.lstrip().rstrip()
                    resultObj['team2-id'] = _findTeamId(res.find_all("td", {"class": "team-cell"})[1].text.lstrip().rstrip())
                    resultObj['team2score'] = converters.to_int(res.find("td", {"class": "result-score"}).find_all("span")[1].text.lstrip().rstrip())
                else:
                    resultObj['team1'] = None
                    resultObj['team1-id'] = None
                    resultObj['team1score'] = None
                    resultObj['team2'] = None
                    resultObj['team2-id'] = None
                    resultObj['team2score'] = None

                if(res.find_all("td", {"class": "star-cell"})):
                    resultObj['stars'] = len(res.find_all("i"))
                    

                results_list.append(resultObj)
                i += 1

    return results_list

def get_match_url(match_id, team1, team2, event):
    team1 = team1.replace(" ", "-")
    team2 = team2.replace(" ", "-")
    event = event.replace(" ", "-")
    return "https://www.hltv.org/matches/" + str(match_id) + "/" + team1 + "-vs-" + team2 + "-" + event
    
def get_matches_url(limit, start_date=None, end_date=None, stars=None):
    urls = []

    lim = 1000
    if limit:
        lim = limit

    for newset in range(0, lim, 100):
        url = f"https://www.hltv.org/results?offset={newset}"
        
        query_params = []
        if start_date:
            query_params.append(f"startDate={start_date}")
        
        if end_date:
            query_params.append(f"endDate={end_date}")
        
        if stars:
            query_params.append(f"stars={stars}")
        
        if query_params:
            url += "&" + "&".join(query_params)
        
        urls.append(url)
    
    return urls

def get_detailed_results_sup(match_id, url, event, stars, match_soup):
    match_details = {}

    # Extract URL
    match_details['url'] = url

    # Extract match id
    match_details['match-id'] = match_id

    # Extract event
    match_details['event'] = event

    # Extract stars
    match_details['stars'] = stars
    
    # Extract date
    date_element = match_soup.find('div', class_='timeAndEvent')
    date_str = date_element.find('div', class_='date').text.strip()
    date_str = re.sub(r'(\d+)(st|nd|rd|th)', r'\1', date_str)
    date_obj = parser.parse(date_str)
    
    # Format the date
    formatted_date = date_obj.strftime("%m/%d/%Y")
    match_details['date'] = formatted_date if date_element else 'N/A'

    # Extract teams
    teams = match_soup.find_all('div', class_='teamName')
    match_details['teams'] = [team.text.strip() for team in teams[:2]]
    
    # Extract score and determine the overall winning team
    score_element = match_soup.find('div', class_='score')
    score_text = score_element.text.strip().replace('\n', '').replace(' ', '') if score_element else 'N/A'
    match_details['score'] = score_text

    # Determine the winning team
    team1_score = int(score_text.split(':')[0])
    team2_score = int(score_text.split(':')[1])
    if team1_score > team2_score:
        match_details['winning_team'] = match_details['teams'][0]
    else:
        match_details['winning_team'] = match_details['teams'][1]
    
    # Extract players and performance
    players = []
    player_tables = match_soup.find_all('table', class_='table totalstats')

    # Extract maps and detailed scores (including first and second halves)
    maps = []
    map_elements = match_soup.find_all('div', class_='mapholder')
    
    player_table_counter = 0  # Counter to track which player table belongs to each map

    for map_element in map_elements:
        map_details = {}
        map_name = map_element.find('div', class_='mapname')
        map_score = map_element.find('div', class_='results')

        # Extract map name
        map_details['map'] = map_name.text.strip() if map_name else 'N/A'
        
        # Initialize team scores with default values in case they don't get assigned later
        team1_total = 0
        team2_total = 0

        if map_score:
            # Parse the score format like "MOUZ6STATS(5:7;1:6)Rooster13"
            map_score_text = map_score.text.strip().replace('\n', '').replace(' ', '')

            # Extract first and second half scores from the "STATS" section
            if "STATS" in map_score_text:
                half_scores = map_score_text.split("STATS")[1].strip('()').split(';')
                map_details['1sthalf'] = half_scores[0] if len(half_scores) > 0 else "N/A"
                
                # Use regex to remove any non-numeric characters from second half score
                second_half_cleaned = re.sub(r'[^\d:]', '', half_scores[1].split(")")[0]) if len(half_scores) > 1 else "N/A"
                map_details['2ndhalf'] = second_half_cleaned

                # Now, extract the numeric values from the first and second halves
                first_half_scores = list(map(int, re.findall(r'\d+', map_details['1sthalf'])))
                second_half_scores = list(map(int, re.findall(r'\d+', map_details['2ndhalf'])))

                if len(first_half_scores) == 2 and len(second_half_scores) == 2:
                    # Add the scores for each team
                    team1_total = first_half_scores[0] + second_half_scores[0]
                    team2_total = first_half_scores[1] + second_half_scores[1]
                    map_details['score'] = f"{team1_total}:{team2_total}"
                else:
                    map_details['score'] = "N/A"

            else:
                map_details['1sthalf'] = "N/A"
                map_details['2ndhalf'] = "N/A"
                map_details['score'] = "N/A"

            # Determine the winning team for this map
            if team1_total > team2_total:
                map_details['winning_team'] = match_details['teams'][0]
            elif team1_total < team2_total:
                map_details['winning_team'] = match_details['teams'][1]
            else:
                map_details['winning_team'] = 'N/A'

            # Get player stats for both teams for the current map
            map_players = []
            for team_index in range(2):  # Iterate for both teams
                if player_table_counter < len(player_tables):  # Ensure the counter doesn't exceed available tables
                    team_name = match_details['teams'][team_index]
                    player_table = player_tables[player_table_counter]  # Get the relevant player table for this map and team
                    map_players += pull_table_stats(map_details['map'], team_name, player_table)
                    player_table_counter += 1  # Increment to the next player table after processing

            map_details['players'] = map_players

        maps.append(map_details)

    match_details['maps'] = maps
    
    # Safely iterate over player tables
    for i, table in enumerate(player_tables[:2]):  # Only iterate over the first two tables for the two teams
        team = match_details['teams'][i]  # Use index to assign team to players
        # Assuming there's no map for this general player data, pass 'N/A' for map_name
        players += pull_table_stats(map_name='ALL', team_name=team, player_table=table)
        
    match_details['players'] = players
    
    return match_details
    
def get_detailed_results(stars=None, count=None, startDate=None, endDate=None, append=False):
    games = get_results(stars, count, startDate, endDate)

    # Export games to Excel for reference
    export_to_excel("results", games)

    urls = [get_match_url(game["match-id"], game["team1"], game["team2"], game["event"]) for game in games]

    soups = scrape_urls_concurrently(urls)
    pattern = r"matches/(\d+)/([a-zA-Z0-9-]+)-vs-([a-zA-Z0-9-]+)-(.+)"

    results = []

    existing_urls = get_existing_urls("Outputs/results.json")
    new_urls = []
    for url in urls:
        if url not in existing_urls:
            new_urls.append(url)
    urls = new_urls

    if append:
        file_exists = os.path.exists("Outputs/results.json")
        with open("Outputs/results.json", 'r+' if file_exists else 'w') as json_file:
            if file_exists:
                json_file.seek(0, os.SEEK_END)
                if json_file.tell() > 0:  # File is not empty
                    json_file.seek(0)
                    content = json_file.read().strip()
                    if content.endswith(']'):
                        json_file.seek(0, os.SEEK_END)
                        json_file.seek(json_file.tell() - 1, os.SEEK_SET)  # Remove the closing bracket
                        json_file.write(',\n')  # Add a comma for new entries
                else:
                    json_file.write('[\n')  # File is empty, start JSON array
            else:
                json_file.write('[\n')  # File does not exist, start JSON array

            for i in range(0, len(urls), batch_size):
                batch_urls = urls[i:i + batch_size]
                soups = scrape_urls_concurrently(batch_urls)

                for j, soup in enumerate(soups):
                    match = re.search(pattern, soup[1])
                    if match is not None:
                        match_id = match.group(1)
                        event = match.group(4).replace('-', ' ')

                    try:
                        stars = 0
                        for game in games:
                            if game["match-id"] == int(match_id):
                                stars = game["stars"]
                                break

                        detailed_result = get_detailed_results_sup(match_id, soup[1], event, stars, soup[0])
                        
                        json.dump(detailed_result, json_file, indent=4)
                        if i + j < len(urls) - 1:
                            json_file.write(',\n')  # Comma between JSON objects except for the last one
                    except:
                        print("Failed on link: " + soup[1])
                print(f"Batch {i // batch_size + 1} of {len(urls) // batch_size} completed.")

            json_file.write('\n]')  # Close the JSON array
    else:
        results = []
        for i in range(0, len(urls), batch_size):
            batch_urls = urls[i:i + batch_size]
            soups = scrape_urls_concurrently(batch_urls)

            for soup in soups:
                match = re.search(pattern, soup[1])
                if match is not None:
                    match_id = match.group(1)
                    event = match.group(4).replace('-', ' ')

                try:
                    stars = 0
                    for game in games:
                        if game["match-id"] == int(match_id):
                            stars = game["stars"]
                            break

                    detailed_result = get_detailed_results_sup(match_id, soup[1], event, stars, soup[0])
                    results.append(detailed_result)

                except:
                    print("Failed on link: " + soup[1])

        return results

def pull_table_stats(map_name, team_name, player_table):
    stat_labels = ['K/D', '+/-', 'ADR', 'HS%', 'Rating']  # Define stat labels
    players = []

    rows = player_table.find_all('tr')
    for row in rows[1:]:  # Skip the header row
        player_details = {}

        # Extract the player's name and nickname
        player_info_element = row.find('td', class_='players')
        if player_info_element:
            player_name_text = player_info_element.text.strip()

            # Extract full name and nickname assuming format "FullName 'Nickname' LastName"
            if "'" in player_name_text:
                name_parts = player_name_text.split("'")
                player_details['name'] = f"{name_parts[0].strip()} {name_parts[2].strip().split()[0]}"
                player_details['nickname'] = name_parts[1].strip()
            else:
                player_details['name'] = player_name_text
                player_details['nickname'] = 'Unknown Nickname'
        else:
            player_details['name'] = 'Unknown Player'
            player_details['nickname'] = 'Unknown Nickname'

        # Add team and map details to player stats
        player_details['team'] = team_name
        player_details['map'] = map_name

        # Extract player stats
        player_stats = [stat.text.strip() for stat in row.find_all('td')[1:]]
        labeled_stats = {stat_labels[i]: player_stats[i] for i in range(len(stat_labels))}

        player_details['stats'] = labeled_stats
        players.append(player_details)

    return players


#Events

def get_event_clip_urls(event_id, event_name):
    event_name = event_name.replace(" ", "-")
    url = f"https://www.hltv.org/events/{event_id}/{event_name}"

    soup = get_parsed_page(url)

    clip_data = []

    info = soup.find('div', class_='event-highlights')
    #print(info)
    days = info.find_all('div', class_='highlight-lists-items hidden')
    for day in days:
        clips = day.find_all('div', class_='highlight-video')
        for clip in clips:
            # Initialize a dictionary for each clip
            clip_info = {}

            # Get the Twitch link from the data-embed-url attribute
            twitch_link = clip.get('data-embed-url')
            clip_info['twitch_link'] = twitch_link

            # Get the view count from the data-view-count attribute
            view_count = clip.get('data-view-count')
            clip_info['view_count'] = view_count

            # Get the description (if available)
            description = clip.get('data-description')
            clip_info['description'] = description

            # Append the collected data to the clip_data list
            clip_data.append(clip_info)

    
    
    return clip_data
            

# Helper functions

def get_existing_urls(file_path):
    urls = set()
    try:
        with open(file_path, 'r') as file:
            data = json.load(file)
            for match in data:
                urls.add(match["url"])
    except FileNotFoundError:
        print("File not found. Returning empty set of match IDs.")
    except json.JSONDecodeError:
        print("JSON decode error. Please check the file format.")
    return urls

def convert_ordered_dict_to_dict(ordered_dict):
    """Recursively convert OrderedDict to normal dict."""
    if isinstance(ordered_dict, OrderedDict):
        return {k: convert_ordered_dict_to_dict(v) for k, v in ordered_dict.items()}
    elif isinstance(ordered_dict, list):
        return [convert_ordered_dict_to_dict(item) for item in ordered_dict]
    else:
        return ordered_dict
  
def export_to_excel(name, data):
    # Create the Outputs folder if it doesn't exist
    os.makedirs('Outputs', exist_ok=True)

    # Write each result to a separate Excel file
    file_path = f'Outputs/{name}.xlsx'
    with pd.ExcelWriter(file_path) as writer:
        if isinstance(data, list):
            df = pd.DataFrame(data)
        else:
            df = pd.DataFrame([data])
        df.to_excel(writer, sheet_name=name, index=False)

In [66]:
# HLTV_COOKIE_TIMEZONE = "Europe/Copenhagen"
# HLTV_ZONEINFO=zoneinfo.ZoneInfo(HLTV_COOKIE_TIMEZONE)
# LOCAL_TIMEZONE_NAME = tzlocal.get_localzone_name()
# LOCAL_ZONEINFO = zoneinfo.ZoneInfo(LOCAL_TIMEZONE_NAME)


# def _get_matches_by_team(table):
#     events = table.find_all("tr", {"class": "event-header-cell"})
#     event_matches = table.find_all("tbody")
#     matches = []
#     for i, event in enumerate(events):

#         event_name = event.find("a", {"class": "a-reset"}).text
#         rows = event_matches[i]("tr", {"class": "team-row"})

#         for row in rows[0:len(rows)]:
#             match = {}
#             dateArr = (row.find(
#                 "td", {"class": "date-cell"}).find("span").text).split('/')

#             dateTextFromArrPadded = _padIfNeeded(dateArr[2]) + "-" + _padIfNeeded(dateArr[1]) + "-" + _padIfNeeded(dateArr[0])

#             dateFromHLTV = datetime.datetime.strptime(dateTextFromArrPadded,'%Y-%m-%d').replace(tzinfo=HLTV_ZONEINFO)
#             dateFromHLTV = dateFromHLTV.astimezone(LOCAL_ZONEINFO)

#             date = dateFromHLTV.strftime('%Y-%m-%d')
#             match['date'] = date
#             match['teams'] = {}

#             if (row.find(
#                 "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-1"})):
#                 match['teams']["team_1"] = row.find(
#                     "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-1"}).text
#                 match['teams']["team_1_id"] = _findTeamId(row.find( "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-1"}).text)
#             else:
#                 match['teams']["team_1"] = None
#                 match['teams']["team_1_id"] = None

#             if (row.find(
#                 "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-2"})):
#                 match['teams']["team_2"] = row.find(
#                     "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-2"}).text
#                 match['teams']["team_2_id"] = _findTeamId(row.find( "td", {"class": "team-center-cell"}).find("a", {"class": "team-name team-2"}).text)
#             else:
#                 match['teams']["team_2"] = None
#                 match['teams']["team_2_id"] = None

#             match["confront_name"] = match['teams']["team_1"] or "TBD" + \
#                 " X " + match['teams']["team_2"] or "TBD"
#             match["championship"] = event_name
#             match_url = row.find(
#                 "td", {"class": "matchpage-button-cell"}).find("a")['href']
#             match['match_id'] = converters.to_int(match_url.split("/")[-2])
#             match['url'] = "https://www.hltv.org" + match_url
#             match['time'] = get_parsed_page("https://www.hltv.org" + match_url).find(
#                 'div', {"class": "timeAndEvent"}).find('div', {"class": "time"}).text
#             matches.append(match)

#     return matches

# def get_results_by_date(start_date, end_date):
#     # Dates like yyyy-mm-dd  (iso)
#     results_list = []
#     offset = 0
#     # Loop through all stats pages
#     while True:
#         url = "https://www.hltv.org/stats/matches?startDate="+start_date+"&endDate="+end_date+"&offset="+str(offset)

#         results = get_parsed_page(url)

#         # Total amount of results of the query
#         amount = int(results.find("span", attrs={"class": "pagination-data"}).text.split("of")[1].strip())

#         # All rows (<tr>s) of the match table
#         pastresults = results.find("tbody").find_all("tr")

#         # Parse each <tr> element to a result dictionary
#         for result in pastresults:
#             team_cols = result.find_all("td", {"class": "team-col"})
#             t1 = team_cols[0].find("a").text
#             t1_id = _findTeamId(team_cols[0].find("a").text)
#             t2 = team_cols[1].find("a").text
#             t2_id = _findTeamId(team_cols[1].find("a").text)
#             t1_score = int(team_cols[0].find_all(attrs={"class": "score"})[0].text.strip()[1:-1])
#             t2_score = int(team_cols[1].find_all(attrs={"class": "score"})[0].text.strip()[1:-1])
#             map = result.find(attrs={"class": "statsDetail"}).find(attrs={"class": "dynamic-map-name-full"}).text
#             event = result.find(attrs={"class": "event-col"}).text
#             dateText = result.find(attrs={"class": "date-col"}).find("a").find("div").text
#             url = "https://hltv.org" + result.find(attrs={"class": "date-col"}).find("a").get("href")
#             match_id = converters.to_int(url.split("/")[-2])
#             dateArr = dateText.split("/")
#             # TODO: yes, this shouldn't be hardcoded, but I'll be very surprised if this API is still a thing in 21XX
#             startingTwoDigitsOfYear = "20"
#             dateTextFromArrPadded = startingTwoDigitsOfYear + _padIfNeeded(dateArr[2]) + "-" + _padIfNeeded(dateArr[1]) + "-" + _padIfNeeded(dateArr[0])

#             dateFromHLTV = datetime.datetime.strptime(dateTextFromArrPadded,'%Y-%m-%d').replace(tzinfo=HLTV_ZONEINFO)
#             dateFromHLTV = dateFromHLTV.astimezone(LOCAL_ZONEINFO)

#             date = dateFromHLTV.strftime('%Y-%m-%d')

#             result_dict = {"match-id": match_id, "team1": t1, "team1-id": t1_id, "team2": t2, "team2-id": t2_id, "team1score": t1_score,
#                         "team2score": t2_score, "date": date, "map": map, "event": event, "url": url}

#             # Add this pages results to the result list
#             results_list.append(result_dict)

#         # Get the next 50 results (next page) or break
#         if offset < amount:
#             offset += 50
#         else:
#             break

#     return results_list

# def top5teams():
#     home = get_parsed_page("https://hltv.org/")
#     teams = []
#     for team in home.find_all("div", {"class": ["col-box rank"], }):
#         team = {'id': _findTeamId(team.text[3:]), 'name': team.text[3:], 'url': "https://hltv.org" + team.find_all("a")[1]["href"]}
#         teams.append(team)
#     return teams

# def top30teams():
#     page = get_parsed_page("https://www.hltv.org/ranking/teams/")
#     teams = page.find("div", {"class": "ranking"})
#     teamlist = []
#     for team in teams.find_all("div", {"class": "ranked-team standard-box"}):
#         newteam = {'name': team.find('div', {"class": "ranking-header"}).select('.name')[0].text.strip(),
#                 'rank': converters.to_int(team.select('.position')[0].text.strip(), regexp=True),
#                 'rank-points': converters.to_int(team.find('span', {'class': 'points'}).text, regexp=True),
#                 'team-id': _findTeamId(team.find('div', {"class": "ranking-header"}).select('.name')[0].text.strip()),
#                 'team-url': "https://hltv.org/team/" + team.find('a', {'class': 'details moreLink'})['href'].split('/')[-1] + "/" + team.find('div', {"class": "ranking-header"}).select('.name')[0].text.strip(),
#                 'stats-url': "https://www.hltv.org" + team.find('a', {'class': 'details moreLink'})['href'],
#                 'team-players': []}
#         for player_div in team.find_all("td", {"class": "player-holder"}):
#             player = {}
#             player['name'] = player_div.find('img', {'class': 'playerPicture'})['title']
#             player['player-id'] = converters.to_int(player_div.select('.pointer')[0]['href'].split("/")[-2])
#             player['url'] = "https://www.hltv.org" + player_div.select('.pointer')[0]['href']
#             newteam['team-players'].append(player)
#         teamlist.append(newteam)
#     return teamlist

# def get_players(teamid):
#     page = get_parsed_page("https://www.hltv.org/?pageid=362&teamid=" + str(teamid))
#     titlebox = page.find("div", {"class": "bodyshot-team"})
#     players = []
#     for player_link in titlebox.find_all("a"):
#         players.append({
#             'id': converters.to_int(player_link["href"].split("/")[2]),
#             'nickname': player_link["title"],
#             'name': player_link.find("img")['title'],
#             'url': "https://hltv.org" + player_link["href"]
#         })

#     return players

# def get_team_info(teamid):
#     """
#     :param teamid: integer (or string consisting of integers)
#     :return: dictionary of team

#     example team id: 5378 (virtus pro)
#     """
#     page = get_parsed_page("https://www.hltv.org/?pageid=179&teamid=" + str(teamid))

#     team_info = {}
#     team_info['team-name'] = page.find("div", {"class": "context-item"}).text
    
#     team_info['team-id'] = _findTeamId(page.find("div", {"class": "context-item"}).text)

#     match_page = get_parsed_page("https://www.hltv.org/team/" + str(teamid) +
#                                 "/" + str(team_info['team-name']) + "#tab-matchesBox")
#     has_not_upcomming_matches = match_page.find(
#         "div", {"class": "empty-state"})
#     if has_not_upcomming_matches:
#         team_info['matches'] = []
#     else:
#         match_table = match_page.find(
#             "table", {"class": "table-container match-table"})
#         team_info['matches'] = _get_matches_by_team(match_table)

#     current_lineup = _get_current_lineup(page.find_all("div", {"class": "col teammate"}))
#     team_info['current-lineup'] = current_lineup

#     historical_players = _get_historical_lineup(page.find_all("div", {"class": "col teammate"}))
#     team_info['historical-players'] = historical_players

#     team_stats_columns = page.find_all("div", {"class": "columns"})
#     team_stats = {}

#     for columns in team_stats_columns:
#         stats = columns.find_all("div", {"class": "col standard-box big-padding"})

#         for stat in stats:
#             stat_value = stat.find("div", {"class": "large-strong"}).text
#             stat_title = stat.find("div", {"class": "small-label-below"}).text
#             team_stats[stat_title] = stat_value

#     team_info['stats'] = team_stats

#     team_info['url'] = "https://hltv.org/stats/team/" + str(teamid) + "/" + str(team_info['team-name'])

#     return team_info

# def _get_current_lineup(player_anchors):
#     """
#     helper function for function above
#     :return: list of players
#     """
#     players = []
#     for player_anchor in player_anchors[0:5]:
#         player = {}
#         buildName = player_anchor.find("img", {"class": "container-width"})["alt"].split('\'')
#         player['country'] = player_anchor.find("div", {"class": "teammate-info standard-box"}).find("img", {"class": "flag"})["alt"]
#         player['name'] = buildName[0].rstrip() + buildName[2]
#         player['nickname'] = player_anchor.find("div", {"class": "teammate-info standard-box"}).find("div", {"class": "text-ellipsis"}).text
#         player['maps-played'] = int(re.search(r'\d+', player_anchor.find("div", {"class": "teammate-info standard-box"}).find("span").text).group())
#         player['url'] = "https://hltv.org" + player_anchor.find("div", {"class": "teammate-info standard-box"}).find("a").get("href")
#         player['id'] = converters.to_int(player_anchor.find("div", {"class": "teammate-info standard-box"}).find("a").get("href").split("/")[-2])
#         players.append(player)
#     return players

# def _get_historical_lineup(player_anchors):
#     """
#     helper function for function above
#     :return: list of players
#     """
#     players = []
#     for player_anchor in player_anchors[5::]:
#         player = {}
#         buildName = player_anchor.find("img", {"class": "container-width"})["alt"].split('\'')
#         player['country'] = player_anchor.find("div", {"class": "teammate-info standard-box"}).find("img", {"class": "flag"})["alt"]
#         player['name'] = buildName[0].rstrip() + buildName[2]
#         player['nickname'] = player_anchor.find("div", {"class": "teammate-info standard-box"}).find("div", {"class": "text-ellipsis"}).text
#         player['maps-played'] = int(re.search(r'\d+', player_anchor.find("div", {"class": "teammate-info standard-box"}).find("span").text).group())
#         player['url'] = "https://hltv.org" + player_anchor.find("div", {"class": "teammate-info standard-box"}).find("a").get("href")
#         player['id'] = converters.to_int(player_anchor.find("div", {"class": "teammate-info standard-box"}).find("a").get("href").split("/")[-2])
#         players.append(player)
#     return players

# def _generate_countdown(date: str, time: str):
#     timenow = datetime.datetime.now().astimezone(LOCAL_ZONEINFO).strftime('%Y-%m-%d %H:%M')
#     deadline = date + " " + time
#     currentTime = datetime.datetime.strptime(timenow,'%Y-%m-%d %H:%M')
#     ends = datetime.datetime.strptime(deadline, '%Y-%m-%d %H:%M')
#     if currentTime < ends:
#         return str(ends - currentTime)
#     return None

# MATCH_WITH_COUNTDOWN = None
# def get_matches():
#     global MATCH_WITH_COUNTDOWN
#     matches = get_parsed_page("https://www.hltv.org/matches/")
#     matches_list = []

#     matchdays = matches.find_all("div", {"class": "upcomingMatchesSection"})

#     for match in matchdays:
#         matchDetails = match.find_all("div", {"class": "upcomingMatch"})
#         date = match.find({'div': {'class': 'matchDayHeadline'}}).text.split()[-1]
#         for getMatch in matchDetails:
#             matchObj = {}

#             matchObj['url'] = "https://hltv.org" + getMatch.find("a", {"class": "match a-reset"}).get("href")
#             matchObj['match-id'] = converters.to_int(getMatch.find("a", {"class": "match a-reset"}).get("href").split("/")[-2])

#             if (date and getMatch.find("div", {"class": "matchTime"})):
#                 timeFromHLTV = datetime.datetime.strptime(date + " " + getMatch.find("div", {"class": "matchTime"}).text,'%Y-%m-%d %H:%M').replace(tzinfo=HLTV_ZONEINFO)
#                 timeFromHLTV = timeFromHLTV.astimezone(LOCAL_ZONEINFO)
#                 matchObj['date'] = timeFromHLTV.strftime('%Y-%m-%d')
#                 matchObj['time'] = timeFromHLTV.strftime('%H:%M')

#                 matchObj['countdown'] = _generate_countdown(date, getMatch.find("div", {"class": "matchTime"}).text)

#                 if not MATCH_WITH_COUNTDOWN and matchObj['countdown']:
#                     MATCH_WITH_COUNTDOWN = converters.to_int(getMatch.find("a", {"class": "match a-reset"}).get("href").split("/")[-2])

#             if getMatch.find("div", {"class": "matchEvent"}):
#                 matchObj['event'] = getMatch.find("div", {"class": "matchEvent"}).text.strip()
#             else:
#                 matchObj['event'] = getMatch.find("div", {"class": "matchInfoEmpty"}).text.strip()

#             if (getMatch.find_all("div", {"class": "matchTeams"})):
#                 matchObj['team1'] = getMatch.find_all("div", {"class": "matchTeam"})[0].text.lstrip().rstrip()
#                 matchObj['team1-id'] = _findTeamId(getMatch.find_all("div", {"class": "matchTeam"})[0].text.lstrip().rstrip())
#                 matchObj['team2'] = getMatch.find_all("div", {"class": "matchTeam"})[1].text.lstrip().rstrip()
#                 matchObj['team2-id'] = _findTeamId(getMatch.find_all("div", {"class": "matchTeam"})[1].text.lstrip().rstrip())
#             else:
#                 matchObj['team1'] = None
#                 matchObj['team1-id'] = None
#                 matchObj['team2'] = None
#                 matchObj['team2-id'] = None

#             matches_list.append(matchObj)

#     return matches_list

# def get_match_countdown(match_id):
#     url = "https://www.hltv.org/matches/" + str(match_id) + "/page"
#     match_page = get_parsed_page(url)
#     timeAndEvent = match_page.find("div", {"class": "timeAndEvent"})
#     date = timeAndEvent.find("div", {"class": "date"}).text
#     time = timeAndEvent.find("div", {"class": "time"}).text
#     dateArr = date.replace("th of","").replace("rd of","").replace("st of","").replace("nd of","").split()
#     dateTextFromArrPadded = _padIfNeeded(dateArr[2]) + "-" + _padIfNeeded(_monthNameToNumber(dateArr[1])) + "-" + _padIfNeeded(dateArr[0])

#     dateFromHLTV = datetime.datetime.strptime(dateTextFromArrPadded,'%Y-%m-%d').replace(tzinfo=HLTV_ZONEINFO)
#     dateFromHLTV = dateFromHLTV.astimezone(LOCAL_ZONEINFO)

#     date = dateFromHLTV.strftime('%Y-%m-%d')

#     return _generate_countdown(date, time)

In [62]:
from pathlib import Path
import shutil

def get_twitch_clip_url(url):
    # Set up the Firefox options for headless mode
    options = Options()
    options.add_argument("--headless")

    driver = webdriver.Firefox(options=options)
    
    driver.get(url)

    time.sleep(random.uniform(1, 2)) 

    # Get the full page source
    page_source = driver.page_source

    # Use regex to find all URLs in the page source (this finds any URLs)
    urls = re.findall(r'https?://[^\s]+', page_source)

    # Filter for the specific kind of clip URL if necessary
    clip_urls = [url for url in urls if "clips.twitch.tv" in url]

    # Print the found URLs
    for clip_url in clip_urls:
        print(f"Found Clip URL: {clip_url}")

    driver.save_screenshot("datacamp.png")

    driver.quit()

    return clip_url

def convert_embed_to_share(embed_url):
    # Split the embedded URL to get the unique clip ID
    clip_id = embed_url.split("clip=")[-1].split("&")[0]
    
    # Construct the share URL
    share_url = f"https://clips.twitch.tv/{clip_id}?tt_content=channel_name&tt_medium=embed"
    
    print(f"Converted Clip URL: {share_url}")

    return share_url

def ssstwitch_download(twitch_url):
    url = "https://ssstwitch.com/"
    download_path = 'C:/Users/Cam/Desktop/Clips'

    options = Options()
    options.add_argument("--headless")
    driver = webdriver.Firefox(options=options)
    driver.set_page_load_timeout(10)
    
    driver.get(url)

    driver.find_element(By.CSS_SELECTOR, "#url").send_keys(twitch_url)

    driver.find_element(By.CSS_SELECTOR, "#downloadBtn").send_keys(Keys.ENTER)

    time.sleep(random.uniform(3, 4))

    driver.find_element(By.XPATH, "//*[@id='video-tab-pane']/table/tbody/tr[3]/td[3]/a").click()
    
    print("Downloading...")

    time.sleep(5) 

    print("Done Downloading")


def move_file(description):
    # Get the current time
    now = datetime.datetime.now()

    # Set source and target directories
    directory = r"C:\Users\Cam\Downloads"
    t_dir = r"C:\Users\Cam\VSCode Projects\HLTVScraper\Outputs\Clips"

    # Sanitize the description to use as the filename (removing or replacing invalid characters)
    safe_description = "".join(c if c.isalnum() or c in (' ', '_', '-') else '_' for c in description)

    # Iterate through files in the source directory
    for file in os.listdir(directory):
        # Full file path
        f = os.path.join(directory, file)

        # Check if it's a file and not a directory
        if os.path.isfile(f):
            # Get file creation time
            create_time = os.path.getctime(f)
            create_date = datetime.datetime.fromtimestamp(create_time)

            # Check if the file was created within the last hour
            time_diff = now - create_date
            if time_diff.total_seconds() <= 3600:  # 3600 seconds = 1 hour
                # Create new filename using the sanitized description
                name, ext = os.path.splitext(file)
                new_file_name = f"{safe_description}{ext}"
                destination = os.path.join(t_dir, new_file_name)

                # Handle potential file name conflicts (append a counter if the file exists)
                counter = 1
                while os.path.exists(destination):
                    destination = os.path.join(t_dir, f"{safe_description}_{counter}{ext}")
                    counter += 1

                # Move the file to the target directory with the new name
                shutil.move(f, destination)
                print(f"Moving {f} to {destination}")
                break  # Move only one file and exit the loop

In [63]:
#result = get_event_clip_urls(7441, "ESL Pro League Season 20")

# for res in result:
#     # Check if 'view_count' is missing or None, set it to 0 in those cases
#     if res is None or 'view_count' not in res:
#         res['view_count'] = 0
#     else:
#         # Ensure 'view_count' is an integer
#         res['view_count'] = converters.to_int(res['view_count'])

# result.sort(key=lambda x: x['view_count'], reverse=True)

#result = get_results(stars=1)

result = get_detailed_results(stars=1, startDate="2023-09-26", endDate="2024-09-26", append=True)

#result = get_top_player_stats()

#result = get_players_stats(startDate='2024-08-23', endDate='2024-09-23', count=100)

#result = get_players_stats()


#800
#get_players_stats(append=True)

# result = get_player_stats(11893, "zywoo")

#pprint.pprint(result)

(1/1) - Successfully scraped https://www.hltv.org/results?offset=0&startDate=2023-09-26&endDate=2024-09-26&stars=1
(1/10) - Successfully scraped https://www.hltv.org/matches/2375770/Natus-Vincere-vs-G2-BLAST-Premier-Fall-Final-2024
(2/10) - Successfully scraped https://www.hltv.org/matches/2375771/Astralis-vs-Spirit-BLAST-Premier-Fall-Final-2024
(3/10) - Successfully scraped https://www.hltv.org/matches/2375772/Vitality-vs-Liquid-BLAST-Premier-Fall-Final-2024
(4/10) - Successfully scraped https://www.hltv.org/matches/2375769/Falcons-vs-FaZe-BLAST-Premier-Fall-Final-2024
(5/10) - Successfully scraped https://www.hltv.org/matches/2375772/Vitality-vs-Liquid-BLAST-Premier-Fall-Final-2024
(6/10) - Successfully scraped https://www.hltv.org/matches/2375771/Astralis-vs-Spirit-BLAST-Premier-Fall-Final-2024
(7/10) - Successfully scraped https://www.hltv.org/matches/2375786/MIBR-vs-BESTIA-CBCS-Masters-2024
(8/10) - Successfully scraped https://www.hltv.org/matches/2375770/Natus-Vincere-vs-G2-BLAS

In [55]:
with open("Outputs/results.json", 'w') as json_file:
    json.dump(result, json_file, indent=4)

In [17]:
count = 15
urls = []  
descriptions = [] 
views = []


for i in range(count):
    urls.append(result[i]['twitch_link'])
    descriptions.append(result[i]['description'].split(" | ")[1])
    views.append(result[i]['view_count'])

twitch_urls = []
for url in urls:
    twitch_urls.append(convert_embed_to_share(url))

df = pd.DataFrame({
    'Twitch Links': twitch_urls,
    'Description': descriptions,
    'Views': views
})

df.to_csv("Outputs/twitch_links.csv", index=False)


Converted Clip URL: https://clips.twitch.tv/AmazonianColdJackalResidentSleeper-ZvlYZRHVqDFYXMHo?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/SmoothNeighborlyZucchiniTooSpicy-d6Zpqc0KvMv3en_G?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/TolerantSaltyHerdPJSalt-ztDZfht3_NYwLoJE?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/SuccessfulWonderfulLobsterAMPEnergy-81u5dxkluWjux7ew?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/BeautifulMotionlessSlothMVGame-Wo9oIDC4qTC21aQX?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/DoubtfulProudCarabeefResidentSleeper-S1oikDHiPPqGbr3x?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/CleanEasyQuailArgieB8-AX55jgoCn9SSRg_2?tt_content=channel_name&tt_medium=embed
Converted Clip URL: https://clips.twitch.tv/OilyTriumphantDuckOSsloth-kAM9z

In [20]:
twitch_urls = []
descriptions = []
file_path = "Outputs/twitch_links.csv"

# Read the CSV file
reader = pd.read_csv(file_path)

# Iterate over each row and extract the URLs and descriptions
for index, row in reader.iterrows():
    twitch_urls.append(row['Twitch Links'])
    descriptions.append(str(row['Views']) + " - " + row['Description'])

downloaded = []

downloaded = pd.read_csv("Outputs/Clips/downloaded.csv")['Downloaded'].tolist()

# Download and move each file
for i, url in enumerate(twitch_urls):
    if url not in downloaded:
        try:
            print(f"Downloading: {url}")
            ssstwitch_download(url)  # Assuming ssstwitch_download downloads the video to the Downloads folder
            print(f"Downloaded {url}")
            
            time.sleep(2)  # Small delay to ensure the file is saved before moving
            
            # Move the file using the corresponding description
            move_file(descriptions[i])

            downloaded.append(url)
        except Exception as e:
            print(f"Error processing {url}: {e}")
    else:
        print(f"File already downloaded: {url}")
pd.DataFrame(downloaded, columns=['Downloaded']).to_csv("Outputs/Clips/downloaded.csv", index=False)

File already downloaded: https://clips.twitch.tv/AmazonianColdJackalResidentSleeper-ZvlYZRHVqDFYXMHo?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/SmoothNeighborlyZucchiniTooSpicy-d6Zpqc0KvMv3en_G?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/TolerantSaltyHerdPJSalt-ztDZfht3_NYwLoJE?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/SuccessfulWonderfulLobsterAMPEnergy-81u5dxkluWjux7ew?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/BeautifulMotionlessSlothMVGame-Wo9oIDC4qTC21aQX?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/DoubtfulProudCarabeefResidentSleeper-S1oikDHiPPqGbr3x?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.twitch.tv/CleanEasyQuailArgieB8-AX55jgoCn9SSRg_2?tt_content=channel_name&tt_medium=embed
File already downloaded: https://clips.t